# Teaching notebook — the ideas behind the match predictor

| | |
|---|---|
| **Purpose** | Show each idea the real pipeline relies on (chronological splits, `shift(1)`, tree splits, ensembling, boosting, log loss) on hand-made toy data small enough to read. |
| **Input** | None — every dataset is typed inline; every cell runs on its own. |
| **Output** | Printed walkthroughs only. The browser version of the same six toys is `docs/playground.html`. |

Each concept is a **markdown cell stating the idea**, then a **code cell demonstrating
exactly that idea** on a toy example you can read at a glance.

This is not the real pipeline. Every dataset is 6–12 rows, made up by hand. Every code cell
is standalone — no project data files, no dependence on cells above it. **Run any section on
its own.**

| § | concept |
|---|---|
| 1 | why a random shuffle leaks and a chronological split doesn't |
| 2 | `shift(1)` then rolling mean |
| 3 | how one decision tree picks a split (Gini, by hand) |
| 4 | what a random forest adds — bootstrap samples and the vote |
| 5 | what gradient boosting does differently — residuals, visibly shrinking |
| 6 | confusion matrix and log loss, computed in numpy |

The final cell maps each toy concept to the exact file and cell in the real project.

## 1. Why a random shuffle leaks

**The idea.** A row's features summarise *matches that came before it*, so rows are not
independent — row 8 carries information about rows 1–7.

A random shuffle scatters rows across train and test regardless of date, putting **later
matches in train and earlier matches in test**. The model learns from the future to predict
the past. A chronological split cuts at one date instead: everything before trains,
everything after tests. That is the real task.

**What the output will show.** Under the shuffle, the train and test date ranges *interleave*
— 2 of the 3 test matches were played before the last training match. Under the
chronological cut, the ranges touch but never overlap.

In [ ]:
import numpy as np
import pandas as pd

# --- toy data: 10 matches, in date order ---
toy = pd.DataFrame({
    "match":  [f"m{i}" for i in range(1, 11)],
    "date":   pd.to_datetime([
        "2024-08-10", "2024-09-14", "2024-10-05", "2024-11-02", "2024-12-07",
        "2025-01-11", "2025-02-15", "2025-03-08", "2025-04-12", "2025-05-17"]),
    "home":   ["ARS", "MUN", "ARS", "CHE", "MUN", "ARS", "CHE", "MUN", "ARS", "CHE"],
    "away":   ["CHE", "ARS", "MUN", "ARS", "CHE", "MUN", "ARS", "CHE", "MUN", "ARS"],
    "result": ["H", "A", "H", "D", "H", "H", "A", "D", "H", "A"],
})
print("the 10 toy matches, in date order")
print(toy.to_string(index=False))

# --- split A: random shuffle, 70/30 ---
rng = np.random.default_rng(7)
shuffled = rng.permutation(len(toy))
cut = int(0.7 * len(toy))
tr_a = toy.iloc[np.sort(shuffled[:cut])]
te_a = toy.iloc[np.sort(shuffled[cut:])]

print("\n" + "=" * 64)
print("SPLIT A — random shuffle")
print(f"  train: {list(tr_a.match)}   {tr_a.date.min().date()} -> {tr_a.date.max().date()}")
print(f"  test : {list(te_a.match)}   {te_a.date.min().date()} -> {te_a.date.max().date()}")
n_leak = int((te_a.date < tr_a.date.max()).sum())
print(f"\n  latest TRAIN date  : {tr_a.date.max().date()}")
print(f"  earliest TEST date : {te_a.date.min().date()}")
print(f"  --> {n_leak} of {len(te_a)} test matches happened BEFORE the last training match.")
print("      The model trains on the future to predict the past. That is the leak.")

# --- split B: chronological, same proportion ---
tr_b, te_b = toy.iloc[:cut], toy.iloc[cut:]

print("\n" + "=" * 64)
print("SPLIT B — chronological")
print(f"  train: {list(tr_b.match)}   {tr_b.date.min().date()} -> {tr_b.date.max().date()}")
print(f"  test : {list(te_b.match)}   {te_b.date.min().date()} -> {te_b.date.max().date()}")
n_leak_b = int((te_b.date < tr_b.date.max()).sum())
print(f"\n  latest TRAIN date  : {tr_b.date.max().date()}")
print(f"  earliest TEST date : {te_b.date.min().date()}")
print(f"  --> {n_leak_b} test matches happened before the last training match. No overlap.")

print("\n  The leak is worse than it looks: if m9 is in train and m3 in test, the model")
print("  has already seen a summary of m3's consequences while being asked to predict m3.")

## 2. `shift(1)` then rolling mean

**The idea.** "Form over the last 3 matches" must mean the 3 matches *before* this one.
Pandas' `rolling(3)` includes the current row, so used directly it feeds a match's own result
into its own feature — the answer becomes an input.

The fix is one operation in one order: **`shift(1)` first, then `rolling`**. Shifting moves
every value down a row, so the window then lands on prior matches only.

**What the output will show.** For `m5` (which won, 3 points), the wrong window is
`[3, 0, 3]` → mean **2.0**, inflated by m5's own 3. The correct window is `[0, 3, 0]` →
mean **1.0**, using only m2–m4. `m1` stays `NaN` under the correct version because it
genuinely has no history.

In [ ]:
import numpy as np
import pandas as pd

# --- toy data: one team, 6 matches in date order ---
hist = pd.DataFrame({
    "match":    ["m1", "m2", "m3", "m4", "m5", "m6"],
    "opponent": ["CHE", "MUN", "LIV", "TOT", "EVE", "NEW"],
    "goals":    [1, 0, 2, 1, 4, 2],
    "points":   [1, 0, 3, 0, 3, 3],      # draw, loss, win, loss, win, win
})
print("one team's history")
print(hist.to_string(index=False))

# --- the shift ---
hist["points_shifted"] = hist["points"].shift(1)
print("\n" + "=" * 68)
print("STEP 1 — shift(1): every value moves down one row")
print(hist[["match", "points", "points_shifted"]].to_string(index=False))
print("\n  m1 has no prior match, so its shifted value is NaN.")
print("  m4's shifted value is 3 -- that is m3's points, now sitting on m4's row.")

# --- rolling, wrong way and right way ---
W = 3
hist["roll_WRONG"] = hist["points"].rolling(W, min_periods=1).mean()
hist["roll_RIGHT"] = hist["points"].shift(1).rolling(W, min_periods=1).mean()

print("\n" + "=" * 68)
print("STEP 2 — rolling mean of 3, without vs with the shift")
print(hist[["match", "points", "roll_WRONG", "roll_RIGHT"]].round(3).to_string(index=False))

# --- the actual window contents for one row ---
row = 4                                        # m5, 0-indexed
pts = hist["points"].tolist()
win_wrong = pts[max(0, row - W + 1): row + 1]  # includes row itself
win_right = pts[max(0, row - W): row]          # stops before row

print("\n" + "=" * 68)
print(f"STEP 3 — what the window holds for row {row} (m5, points = {pts[row]})")
print(f"  WRONG  window = {win_wrong}  ->  mean {np.mean(win_wrong):.3f}"
      f"   <-- contains m5's own {pts[row]}")
print(f"  RIGHT  window = {win_right}  ->  mean {np.mean(win_right):.3f}"
      f"   <-- only m2, m3, m4")
print(f"\n  The wrong feature reads {np.mean(win_wrong):.1f} instead of "
      f"{np.mean(win_right):.1f} purely because m5 won.")
print("  A model would 'discover' that teams who did well in a match... did well in it.")

## 3. How one decision tree picks a split

**The idea.** A tree sorts a mixed pile of labelled rows into purer piles. Purity needs a
number, and the usual one is **Gini impurity**:

> Gini = 1 − (sum of each class's squared share)

All one class scores 0; an even 3-way mix scores ≈ 0.667. Lower is purer.

To choose a split, the tree tries a threshold, computes each child pile's Gini, takes their
**size-weighted average**, and subtracts that from the parent's Gini. The difference is the
**gain**. It scans every feature at every threshold and keeps the largest gain.

**What the output will show.** The parent Gini is 0.6562. The split `elo_diff <= 40` sends
5 rows left (Gini 0.48) and 3 rows right (Gini 0.00, perfectly pure), for a gain of 0.3562 —
the best of the seven candidates. Thresholds are the **midpoints between observed values**,
which is why 40 appears rather than 20 or 60. Note that the winner is not simply the split
producing the purest single child: weighting by pile size is what stops the tree chasing
tiny pockets.

In [ ]:
import numpy as np
import pandas as pd

# --- toy data: 8 matches, one feature, 3 classes ---
d = pd.DataFrame({
    "elo_diff": [-120, -80, -30, -10, 20, 60, 90, 140],
    "result":   ["A",  "A",  "D",  "A",  "D", "H", "H", "H"],
})
print("the 8 toy matches (sorted by elo_diff)")
print(d.to_string(index=False))


def gini(labels):
    """Return (gini value, printable working)."""
    labels = list(labels)
    n = len(labels)
    if n == 0:
        return 0.0, "empty"
    terms, total = [], 0.0
    for cls in ["H", "D", "A"]:
        c = labels.count(cls)
        total += (c / n) ** 2
        terms.append(f"({c}/{n})^2")
    return 1 - total, f"1 - [{' + '.join(terms)}] = 1 - {total:.4f}"


# --- parent impurity ---
counts = {cls: int((d.result == cls).sum()) for cls in ["H", "D", "A"]}
g_parent, work = gini(d.result)
print("\n" + "=" * 70)
print("PARENT PILE")
print(f"  counts: {counts}   n = {len(d)}")
print(f"  Gini = {work} = {g_parent:.4f}")

# --- one candidate split, worked through ---
THR = 40
left, right = d[d.elo_diff <= THR], d[d.elo_diff > THR]
g_left, w_left = gini(left.result)
g_right, w_right = gini(right.result)
weighted = len(left) / len(d) * g_left + len(right) / len(d) * g_right

print("\n" + "=" * 70)
print(f"CANDIDATE SPLIT:  elo_diff <= {THR}")
print(f"  LEFT  (n={len(left)}): {list(left.result)}")
print(f"        Gini = {w_left} = {g_left:.4f}")
print(f"  RIGHT (n={len(right)}): {list(right.result)}")
print(f"        Gini = {w_right} = {g_right:.4f}")
print(f"\n  weighted child Gini = {len(left)}/{len(d)} * {g_left:.4f}"
      f" + {len(right)}/{len(d)} * {g_right:.4f} = {weighted:.4f}")
print(f"  GAIN = {g_parent:.4f} - {weighted:.4f} = {g_parent - weighted:.4f}")

# --- score every candidate threshold ---
vals = sorted(d.elo_diff.unique())
rows = []
for t in [(a + b) / 2 for a, b in zip(vals, vals[1:])]:      # midpoints
    L, R = d[d.elo_diff <= t], d[d.elo_diff > t]
    gl, _ = gini(L.result)
    gr, _ = gini(R.result)
    wavg = len(L) / len(d) * gl + len(R) / len(d) * gr
    rows.append({"threshold": t, "n_left": len(L), "n_right": len(R),
                 "gini_left": gl, "gini_right": gr,
                 "weighted": wavg, "gain": g_parent - wavg})

scan = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("EVERY CANDIDATE THRESHOLD, SCORED")
print(scan.round(4).to_string(index=False))

best = scan.loc[scan.gain.idxmax()]
print(f"\n  best split: elo_diff <= {best.threshold}   gain {best.gain:.4f}")
print("  That number -- the gain -- is what the tree maximises at every node.")
print("  A real tree now repeats this whole scan inside each child pile.")

### Cross-check against scikit-learn

**The idea.** The hand computation above should give the same threshold and the same root
impurity as a real library implementation, because it is the same arithmetic.

The cell is guarded: on this machine `sklearn.tree` loads, but `sklearn.ensemble` does not
(Windows Smart App Control blocks some scipy binaries), which is why §4 below builds its
trees by hand instead.

In [ ]:
import numpy as np
import pandas as pd

# self-contained: rebuild the same toy data
d = pd.DataFrame({
    "elo_diff": [-120, -80, -30, -10, 20, 60, 90, 140],
    "result":   ["A",  "A",  "D",  "A",  "D", "H", "H", "H"],
})

try:
    from sklearn.tree import DecisionTreeClassifier, export_text

    clf = DecisionTreeClassifier(criterion="gini", max_depth=1, random_state=0)
    clf.fit(d[["elo_diff"]], d["result"])
    print("scikit-learn's depth-1 tree:")
    print(export_text(clf, feature_names=["elo_diff"]))
    print(f"chosen threshold : {clf.tree_.threshold[0]}          (hand result: 40.0)")
    print(f"impurity at root : {clf.tree_.impurity[0]:.4f}      (hand result: 0.6562)")
    print("\nSame threshold, same root impurity. Same arithmetic.")
except Exception as exc:
    print("scikit-learn unavailable here -- skipping the cross-check.")
    print(f"  reason: {type(exc).__name__}: {exc}")
    print("\nThe hand computation above is the authoritative version.")

## 4. What a random forest adds over one tree

**The idea.** One deep tree memorises quirks of the exact rows it saw. A forest builds many
trees, each deliberately handicapped so they *disagree*, then takes a **majority vote**.

Two independent handicaps:

1. **Bootstrap rows** — each tree trains on a sample drawn *with replacement*, so it sees
   roughly two-thirds of the data, a different two-thirds each time.
2. **Random feature subset per split** — each split may only consider a few features, so not
   every tree opens on the same dominant one. (With one feature here, only #1 is visible.)

Averaging works because the trees' *errors* point in different directions and cancel, while
the real signal is what most trees find independently.

**What the output will show.** Five bootstrap samples, each seeing 6–8 of the 10 rows, each
learning a slightly different threshold. Their votes are `D, H, H, H, H` — one tree is wrong,
and the 4-of-5 majority still lands on the correct answer. Had all five trained on identical
data they would have learned an identical rule and the vote would have added nothing. **The
disagreement is the point.**

The trees are built by hand here because `sklearn.ensemble` cannot load on this machine.

In [ ]:
import numpy as np
import pandas as pd

# --- toy data: 10 training matches + 1 holdout ---
train = pd.DataFrame({
    "elo_diff": [-120, -95, -60, -30, -10, 15, 45, 70, 105, 150],
    "result":   ["A",  "A",  "D",  "A",  "D", "H", "D", "H", "H",  "H"],
})
holdout = {"elo_diff": 35, "true_result": "H"}

print("training data (10 rows)")
print(train.to_string(index=False))
print(f"\nholdout match: elo_diff = {holdout['elo_diff']}, "
      f"actual result = {holdout['true_result']}")


def gini(labels):
    n = len(labels)
    if n == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    return 1 - ((counts / n) ** 2).sum()


def best_stump(df):
    """Depth-1 tree: scan midpoint thresholds, return (threshold, left_label, right_label, gain)."""
    y = df.result.to_numpy()
    g_parent = gini(y)
    vals = sorted(df.elo_diff.unique())
    best = None
    for t in [(a + b) / 2 for a, b in zip(vals, vals[1:])]:
        m = df.elo_diff.to_numpy() <= t
        if m.all() or (~m).all():
            continue
        gain = g_parent - (m.mean() * gini(y[m]) + (~m).mean() * gini(y[~m]))
        if best is None or gain > best[3]:
            best = (t, pd.Series(y[m]).mode()[0], pd.Series(y[~m]).mode()[0], gain)
    return best


# --- grow 5 trees on 5 bootstrap samples ---
rng = np.random.default_rng(11)
votes = []
print("\n" + "=" * 74)
for i in range(1, 6):
    idx = rng.integers(0, len(train), len(train))          # WITH replacement
    sample = train.iloc[idx]
    thr, lab_l, lab_r, gain = best_stump(sample)
    vote = lab_l if holdout["elo_diff"] <= thr else lab_r
    votes.append(vote)

    n_unique = len(set(idx.tolist()))
    print(f"TREE {i}")
    print(f"  bootstrap row indices : {idx.tolist()}")
    print(f"  distinct rows seen    : {n_unique} of {len(train)}"
          f"   ({n_unique / len(train):.0%} -- the rest were left out)")
    print(f"  labels in this sample : {list(sample.result)}")
    print(f"  rule learned          : elo_diff <= {thr}  ->  {lab_l}   else  {lab_r}"
          f"    (gain {gain:.4f})")
    print(f"  vote on holdout       : {vote}")
    print()

# --- the vote ---
tally = pd.Series(votes).value_counts()
print("=" * 74)
print("THE VOTE")
print(f"  individual votes: {votes}")
for cls, n in tally.items():
    print(f"    {cls}: {n}/{len(votes)}  = {n / len(votes):.0%}")
print(f"\n  forest prediction : {tally.index[0]}")
print(f"  actual result     : {holdout['true_result']}")
print("\n  The trees disagree; individually one is wrong. The majority is right,")
print("  and only because the bootstrap samples made their errors independent.")

## 5. What gradient boosting does differently

**The idea.** A forest grows its trees **in parallel** and averages them — every tree tackles
the same problem independently.

Boosting grows trees **in sequence**, each trained on the **residuals**: what the model so
far still gets wrong. Tree 2 never sees the original target, only tree 1's leftover error. A
learning rate scales each tree's contribution down so the correction is gradual.

**What the output will show.** Starting from the mean (0.0), the total squared error falls
every round: **10.00 → 4.94 → 2.09 → 1.07**. The residual printed at the top of each round
is literally the input to that round's tree — round 2 sees what round 1 failed to fix. That
is the whole difference from a forest: *forest = many independent guesses averaged;
boosting = one guess, repeatedly corrected.*

In [ ]:
import numpy as np
import pandas as pd

# --- toy data: 6 matches, predict goal margin from elo_diff ---
x = np.array([-100, -60, -20, 10, 50, 90], dtype=float)
y = np.array([  -2,  -1,   0,  0,  1,  2], dtype=float)

print("toy data")
print(pd.DataFrame({"elo_diff": x, "margin": y}).to_string(index=False))

LR, ROUNDS = 0.5, 3


def best_stump_regression(x, resid):
    """Depth-1 regression tree on the residuals: minimise summed squared error."""
    best = None
    vals = sorted(set(x))
    for t in [(a + b) / 2 for a, b in zip(vals, vals[1:])]:
        m = x <= t
        if m.all() or (~m).all():
            continue
        pl, pr = resid[m].mean(), resid[~m].mean()
        sse = ((resid[m] - pl) ** 2).sum() + ((resid[~m] - pr) ** 2).sum()
        if best is None or sse < best[3]:
            best = (t, pl, pr, sse)
    return best


# --- round 0: the baseline guess ---
F = np.full_like(y, y.mean())
print("\n" + "=" * 76)
print(f"ROUND 0 -- baseline prediction = mean(margin) = {y.mean():.3f} for every row")
print(f"  prediction : {np.round(F, 3)}")
print(f"  residual   : {np.round(y - F, 3)}      <-- what tree 1 must learn")
print(f"  total squared error: {((y - F) ** 2).sum():.4f}")

# --- boosting rounds ---
for r in range(1, ROUNDS + 1):
    resid = y - F                                    # what is still wrong
    err_before = (resid ** 2).sum()
    thr, pl, pr, _ = best_stump_regression(x, resid)
    step = np.where(x <= thr, pl, pr)                # this tree's raw output
    F = F + LR * step                                # scaled by the learning rate

    print("\n" + "=" * 76)
    print(f"ROUND {r}")
    print(f"  tree {r} is fit to the residual : {np.round(resid, 3)}")
    print(f"  rule learned   : elo_diff <= {thr}  ->  {pl:+.3f}   else  {pr:+.3f}")
    print(f"  raw tree output                : {np.round(step, 3)}")
    print(f"  scaled by lr={LR}               : {np.round(LR * step, 3)}")
    print(f"  updated prediction             : {np.round(F, 3)}")
    print(f"  NEW residual                   : {np.round(y - F, 3)}")
    print(f"  total squared error: {((y - F) ** 2).sum():.4f}   (was {err_before:.4f})")

print("\n" + "=" * 76)
print("FINAL")
print(f"  actual margin    : {np.round(y, 3)}")
print(f"  model prediction : {np.round(F, 3)}")
print(f"  remaining error  : {np.round(y - F, 3)}")
print("\n  Every round's residual is smaller than the last. After round 1 no tree ever")
print("  saw the target directly -- each saw only the previous model's mistakes.")

### Cross-check against XGBoost

**The idea.** The loop above *is* gradient boosting, so a real library given the same
settings — 3 rounds, learning rate 0.5, depth-1 trees, same starting value, regularisation
switched off — should reproduce it exactly.

**What the output will show.** Every prediction matches to 0.000000. The library is faster
and adds regularisation by default; the algorithm is the one above.

In [ ]:
import numpy as np
import pandas as pd

# self-contained: rebuild the toy data and the hand-rolled boosting
x = np.array([-100, -60, -20, 10, 50, 90], dtype=float)
y = np.array([  -2,  -1,   0,  0,  1,  2], dtype=float)
LR, ROUNDS = 0.5, 3


def best_stump_regression(x, resid):
    best = None
    vals = sorted(set(x))
    for t in [(a + b) / 2 for a, b in zip(vals, vals[1:])]:
        m = x <= t
        if m.all() or (~m).all():
            continue
        pl, pr = resid[m].mean(), resid[~m].mean()
        sse = ((resid[m] - pl) ** 2).sum() + ((resid[~m] - pr) ** 2).sum()
        if best is None or sse < best[3]:
            best = (t, pl, pr, sse)
    return best


F = np.full_like(y, y.mean())
for _ in range(ROUNDS):
    thr, pl, pr, _ = best_stump_regression(x, y - F)
    F = F + LR * np.where(x <= thr, pl, pr)

try:
    import xgboost as xgb

    model = xgb.XGBRegressor(
        n_estimators=ROUNDS, learning_rate=LR, max_depth=1,
        base_score=float(y.mean()),      # same starting point as ours
        reg_lambda=0.0,                  # our version has no regularisation
        min_child_weight=0.0,
        objective="reg:squarederror",
    )
    model.fit(x.reshape(-1, 1), y)
    xgb_pred = model.predict(x.reshape(-1, 1))

    print("hand-rolled boosting vs XGBoost -- same 3 rounds, same learning rate")
    print(pd.DataFrame({
        "elo_diff": x, "actual": y,
        "by_hand": np.round(F, 4),
        "xgboost": np.round(xgb_pred, 4),
        "diff": np.round(F - xgb_pred, 6),
    }).to_string(index=False))
    print(f"\nlargest absolute difference: {np.abs(F - xgb_pred).max():.6f}")
    print("\nIdentical. The loop in the previous cell is what XGBoost is doing,")
    print("just with smarter split-finding and regularisation available.")
except Exception as exc:
    print("xgboost unavailable here -- skipping the cross-check.")
    print(f"  reason: {type(exc).__name__}: {exc}")
    print(f"\nhand-rolled prediction after {ROUNDS} rounds: {np.round(F, 4)}")

## 6. Confusion matrix and log loss, by hand

**The idea.** Accuracy hides *which* class a model fails on. Two tools fix that.

A **confusion matrix** cross-tabulates truth (rows) against prediction (columns). The
diagonal is correct; each off-diagonal cell is one specific kind of mistake. From it you read
**recall** (of the real draws, how many did we catch?) and **precision** (when we said draw,
how often were we right?).

**Log loss** grades the *probabilities*, not the labels: take the probability assigned to the
class that actually happened, apply −log, average. Confident and right ≈ 0; confident and
wrong is punished hard.

**What the output will show.** Accuracy 7/12 = 0.583, but draw recall is only 1/3 — the model
rarely says "D", exactly the pattern the real project shows. Log loss comes out 0.8588
against 1.0986 for a model that always guessed 1/3–1/3–1/3; beat that number or the
probabilities are worthless. The two metrics answer different questions: the matrix asks
*which class am I failing*, log loss asks *are my probabilities honest*.

In [ ]:
import numpy as np
import pandas as pd

LABELS = ["H", "D", "A"]
idx = {lab: i for i, lab in enumerate(LABELS)}

# --- toy predictions: 12 matches ---
y_true = np.array(["H","H","H","H","H","D","D","D","A","A","A","A"])
y_pred = np.array(["H","H","H","A","D","H","A","D","A","A","H","A"])

print("truth vs prediction")
print(pd.DataFrame({"actual": y_true, "predicted": y_pred}).to_string())

# --- confusion matrix, counted explicitly ---
cm = np.zeros((3, 3), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[idx[t], idx[p]] += 1

print("\n" + "=" * 68)
print("CONFUSION MATRIX  (rows = truth, columns = prediction)")
print(pd.DataFrame(cm,
                   index=[f"actual {l}" for l in LABELS],
                   columns=[f"pred {l}" for l in LABELS]).to_string())
print(f"\n  correct = the diagonal = {cm[0,0]} + {cm[1,1]} + {cm[2,2]} = {np.trace(cm)}")
print(f"  accuracy = {np.trace(cm)}/{cm.sum()} = {np.trace(cm) / cm.sum():.4f}")

# --- per-class recall and precision, as fractions ---
print("\n" + "=" * 68)
print("PER-CLASS BREAKDOWN")
for i, lab in enumerate(LABELS):
    tp, row_tot, col_tot = cm[i, i], cm[i, :].sum(), cm[:, i].sum()
    rec = tp / row_tot if row_tot else 0.0
    prec = tp / col_tot if col_tot else 0.0
    print(f"  {lab}:  recall    = {tp}/{row_tot} = {rec:.3f}"
          f"   (of the real {lab}s, how many we caught)")
    print(f"      precision = {tp}/{col_tot} = {prec:.3f}"
          f"   (when we said {lab}, how often right)")

# --- log loss, row by row ---
print("\n" + "=" * 68)
print("LOG LOSS")
proba = np.array([                     # columns in LABELS order: H, D, A
    [0.70, 0.20, 0.10],   # H  confident and right
    [0.55, 0.25, 0.20],   # H
    [0.45, 0.30, 0.25],   # H
    [0.30, 0.25, 0.45],   # H  confident and WRONG
    [0.40, 0.35, 0.25],   # H
    [0.50, 0.30, 0.20],   # D
    [0.25, 0.30, 0.45],   # D
    [0.30, 0.40, 0.30],   # D  hedged toward draw and right
    [0.20, 0.25, 0.55],   # A
    [0.15, 0.20, 0.65],   # A
    [0.45, 0.30, 0.25],   # A  confident and WRONG
    [0.25, 0.25, 0.50],   # A
])

print(f"  {'actual':>7}  {'p(H)':>5} {'p(D)':>5} {'p(A)':>5}   {'p(actual)':>9}  {'-log(p)':>8}")
per_row = []
for t, p in zip(y_true, proba):
    p_true = p[idx[t]]
    loss = -np.log(p_true)
    per_row.append(loss)
    print(f"  {t:>7}  {p[0]:5.2f} {p[1]:5.2f} {p[2]:5.2f}   {p_true:9.2f}  {loss:8.4f}")

per_row = np.array(per_row)
print(f"\n  log loss = mean of the last column = {per_row.sum():.4f} / {len(per_row)}"
      f" = {per_row.mean():.4f}")
print(f"\n  worst row contributed {per_row.max():.4f}  (probability "
      f"{np.exp(-per_row.max()):.2f} on what actually happened)")
print(f"  best  row contributed {per_row.min():.4f}  (probability "
      f"{np.exp(-per_row.min()):.2f})")
print(f"\n  Always guessing 1/3, 1/3, 1/3 scores {-np.log(1/3):.4f}.")
print("  Beat that or the probabilities carry no information.")

## Where each idea lives in the real project

| § toy concept | real file | where |
|---|---|---|
| **1** chronological split | `model.ipynb` | §4 — defines `VALID` / `TEST` / `HOLDOUT`, slices `df` by `Season` |
| **1** leak checking | `features.ipynb` | §3.6 — recomputes a feature independently, asserts it matches |
| **2** `shift(1)` then rolling | `features.ipynb` | §3.2 — `prior_rolling()` / `prior_expanding()` / `prior_ewm()` over `ROLL_STATS` |
| **2** same trick elsewhere | `features.ipynb` | §3.3 venue form & head-to-head, §3.4a rolling shot quality |
| **3** split finding / impurity | `model.ipynb` | §6 — `train_classifier()`; XGBoost runs this scan at every node |
| **4** ensembling & disagreement | `model.ipynb` | §6 — `subsample=0.85`, `colsample_bytree=0.7` are the two randomness knobs |
| **5** boosting rounds | `model.ipynb` | §6 — `eta=0.03`, up to 4000 rounds with early stopping on the validation block |
| **6** confusion matrix, log loss | `pl_model.py` | `confusion()`, `log_loss()`, `class_report()` — hand-rolled in numpy |
| **6** reading the result | `model.ipynb` | §9 test evaluation, §10b the rolling-origin table, §12 the holdout |

Two differences between toy and real worth knowing:

- The project uses **XGBoost, not a random forest** — `sklearn.ensemble` cannot load on this
  machine (Windows Smart App Control blocks some scipy binaries). §4 still matters:
  `subsample` and `colsample_bytree` are the same two ideas inside XGBoost.
- Real features are **home-minus-away differences**, so one match is one row. The toy
  examples skip that reshaping to stay readable; `features.ipynb` §3.1 and §3.5 do it at
  full scale.
